In [25]:
%reset -f
import arcpy
arcpy.ResetEnvironments()
import os
from arcpy.sa import ExtractMultiValuesToPoints
import math

# check out Spatial Analyst
arcpy.CheckOutExtension("Spatial")

'CheckedOut'

In [26]:
# ── PATHS ────────────────────────────────────────────────────────────────────────────────────────────

# Input bathymetry and slope rasters
bathy_p26920_v6642 = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\MASTER_RASTERS_USC_GBC_MERGED.gdb\BATHY_USCGBC_p26920_v6642"
slope_p26920_v6642 = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\MASTER_RASTERS_USC_GBC_MERGED.gdb\SLOPE_USCGBC_p26920_v6642"

# Input USC bathy raster footprint for populating "native_raster_resolution_m" column in point feature class Attribute Tables 
# *** only use footprint from bathy raster - USC bathy/slope rasters have exact same resolution and only creating one raster res column per Attribute Table so output res value for each row will be the same for bathy and slope
usc_fp_p26920_v6642 = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\MASTER_RASTERS_USC_GBC_MERGED.gdb\footprint_USC10m_bathy_p26920_v6642"

# Input HF mission + species "calc_" point feature classes
hf_points_p26920 = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Missions\GIS_UVI2_Around_STX_20250306\GIS_UVI2_Around_STX_20250306.gdb\calc_HF_background_points_p26920"
sperm_points_p26920 = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Missions\GIS_UVI2_Around_STX_20250306\GIS_UVI2_Around_STX_20250306.gdb\calc_sperm_points_p26920"
odont_points_p26920 = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Missions\GIS_UVI2_Around_STX_20250306\GIS_UVI2_Around_STX_20250306.gdb\calc_odontocetes_points_p26920"

In [27]:
# ── SETTINGS ─────────────────────────────────────────────────────────────────────────────────────────
arcpy.env.overwriteOutput = True    # overwrite existing outputs on rerun

In [28]:
############### NOTE ###############
# 'Extract Multi Values to Points' tool edits Attribute Table of input feature class in-place (not additional output files/etc.)
# Edits are saved to source feature class in .gdb (not just to layer on map)
####################################

In [29]:
# ── 1. CALCULATE BATHYMETRIC DEPTH AND SLOPE FOR EACH POINT ──────────────────────────────────────────

# Extract Multi Values to Points (Spatial Analyst Tools): Function
def extract_bathy_slope(in_points, bathy_raster, slope_raster):

    try:
        ExtractMultiValuesToPoints(
            in_point_features=in_points,
            in_rasters=[
                # output values calculated from raster 'bathy_p26920_v6642' to field new field "bathymetric_depth_m"
                [bathy_raster, "bathymetric_depth_m"],
                # output values calculated from raster 'slope_p26920_v6642' to field new field "bathymetric_slope_deg"
                [slope_raster, "bathymetric_slope_deg"],
            ],
            # values calculated from adjacent cells with valid values (NoData values ignored); bathy and slope are continuous variables so calculations across gradient (bilinear) are better representation of data than single value from center of raster grid cell (nearest neighbor)
            bilinear_interpolate_values="BILINEAR"
        )
        # print confirmation that tool ran successfully (* even if some cells in table did NOT recieve a bathy/slope value due to being located on tiny gap between GBC and USC rasters in merged raster used for calculations)
        print(f"✅ Extracted depth and slope values for rows in {os.path.basename(in_points)} \n")

    # print warning message if tool fails
    except Exception as e:
        print(f"❌[FAILURE] Extracting depth/slope values failed for {os.path.basename(in_points)}: {e} \n")


# Call function for all point feature classes
extract_bathy_slope(hf_points_p26920, bathy_p26920_v6642, slope_p26920_v6642)
extract_bathy_slope(sperm_points_p26920, bathy_p26920_v6642, slope_p26920_v6642)
extract_bathy_slope(odont_points_p26920, bathy_p26920_v6642, slope_p26920_v6642)

✅ Extracted depth and slope values for rows in calc_HF_background_points_p26920 

✅ Extracted depth and slope values for rows in calc_sperm_points_p26920 

✅ Extracted depth and slope values for rows in calc_odontocetes_points_p26920 



In [30]:
# Fields added to Attribute Table by 'Extract Multi Values to Points' tool: 
# bathymetric_depth_m, bathymetric_slope_deg

In [31]:
# ── STANDARDIZE EMPTY CELL VALUES TO NULL + WARN IF ANY ROWS MISSING DEPTH/SLOPE DATA ────────────────

# List of fields to clean and check on each point feature class Attribute Table
check_fields = [
    "bathymetric_depth_m",
    "bathymetric_slope_deg"
]


# Function to inspect and clean point feature class Attribute Table (standardize missing data to true Null and report remaining gaps)
def clean_point_table(table, check_fields):

    # get field names present in point fc Attribute Table
    existing = {f.name for f in arcpy.ListFields(table)}
    # find any check_fields missing in table
    absent = [f for f in check_fields if f not in existing]
    # print warning message if any expected fields are missing and skip table
    if absent:
        print(f"⚠️[WARNING] {os.path.basename(table)} missing expected field(s): {', '.join(absent)} - skipping \n")
        return

    # if all check_fields were successfully found in table, standardize missing data cells to true Null
    with arcpy.da.UpdateCursor(table, check_fields) as cursor:
        # loop through each row
        for row in cursor:
            # track whether this row was modified
            changed = False
            # inspect each field value in the row
            for i, value in enumerate(row):
                # replace NaN (Extract Multi Values to Points output for points falling on raster NoData cells) with true Null
                if value is not None and isinstance(value, float) and math.isnan(value):
                    row[i] = None
                    changed = True
                # replace empty/blank strings with true Null
                elif isinstance(value, str) and value.strip() == "":
                    row[i] = None
                    changed = True
            # write fixed value back to row only if something was changed
            if changed:
                cursor.updateRow(row)

    # count and report total n records (rows) in feature class table
    total = int(arcpy.management.GetCount(table)[0])
    print(f"\n------ Report for {os.path.basename(table)} ------\n")
    print(f"  Total rows in Attribute Table: {total}")

    # loop through each of the check_fields (defined above) to report remaining gaps
    for field in check_fields:
        # count rows where value is still Null after cleaning
        null_count = 0
        # collect object IDs of any null rows for inspection
        null_oids = []
        with arcpy.da.SearchCursor(table, ["OID@", field]) as cursor:
            for oid, value in cursor:
                # NoData/unmatched points returned as 'None' or 'NaN'
                if value is None or (isinstance(value, float) and math.isnan(value)):
                    null_count += 1
                    null_oids.append(oid)
        # report presence/absence of Null cells in each check field
        if null_count == 0:
            print(f"  {field}: ✅ No null values in field")
        else:
            print(f"  {field}: ⚠️[WARNING] {null_count} Null value(s) in row(s) with OID: {null_oids}")


# Call function for all point feature classes
clean_point_table(hf_points_p26920, check_fields)
clean_point_table(sperm_points_p26920, check_fields)
clean_point_table(odont_points_p26920, check_fields)


------ Report for calc_HF_background_points_p26920 ------

  Total rows in Attribute Table: 4319
  bathymetric_depth_m: ✅ No null values in field
  bathymetric_slope_deg: ✅ No null values in field

------ Report for calc_sperm_points_p26920 ------

  Total rows in Attribute Table: 358
  bathymetric_depth_m: ✅ No null values in field
  bathymetric_slope_deg: ✅ No null values in field

------ Report for calc_odontocetes_points_p26920 ------

  Total rows in Attribute Table: 590
  bathymetric_depth_m: ✅ No null values in field
  bathymetric_slope_deg: ✅ No null values in field


In [32]:
# ── 2. ADD + POPULATE COLUMN TO TRACK NATIVE RESOLUTION OF RASTER USED FOR BATHY/SLOPE CALCULATIONS ──

# Function to add "native_raster_resolution_m" field to point feature class Attribute Tables, select points inside/outside USC footprint polygon, and populate "native_raster_resolution_m" field based on overlap with USC footprint
def assign_raster_resolution(in_points, footprint_polygon):
    field_name = "native_raster_resolution_m"

    # create field only if it does NOT already exist (** if field IS already present, any data it contains will be overwritten by Calculate Field tool)
    existing_fields = [f.name for f in arcpy.ListFields(in_points)]
    if field_name not in existing_fields:
        # Add Field (Data Management Tools):
        arcpy.management.AddField(
            in_table=in_points,
            field_name=field_name,
            # "SHORT" data type (16-bit signed integer that can store integers within range -32,768 to 32,767)
            field_type="SHORT"
        )

    # create temporary feature layer object from point feature class (* in-memory layer object that can hold selections* created by SelectLayerByLocation/SelectLayerByAttribute tools (source feature class files cannot hold selections) - selections in feature layer determine which rows get written in Attribute Table of original feature class file)
    layer = "points_lyr"
    # Make Feature Layer (Data Management Tools):
    arcpy.management.MakeFeatureLayer(
        in_features=in_points,
        out_layer=layer
    )

    ########################### points WITHIN footprint of USC raster (10-m resolution) ###########################

    # Select Layer by Location (Data Management Tools): Select all points WITHIN (completely or partially) USC footprint
    arcpy.management.SelectLayerByLocation(
        # use feature layer created from point feature class
        in_layer=layer,
        # *"INTERSECT" (default) includes points completely AND partially in USC footprint
        overlap_type="INTERSECT",
        # use USC footprint polygon feature class as location parameter
        select_features=footprint_polygon,
        # "NEW_SELECTION" clears any existing selections in feature layer and starts fresh with only features matching current query parameters
        selection_type="NEW_SELECTION"
    )

    inside_count = int(arcpy.management.GetCount(layer)[0])

    # Calculate Field (Data Management Tools): Populate "native_raster_resolution_m" field of selected rows (within USC fp) with '10' (writes data to original feature class file for ONLY rows held in feature layer selection)
    arcpy.management.CalculateField(
        in_table=layer,
        field=field_name,
        expression=10,
        expression_type="PYTHON3"
    )

    ########################### points OUTSIDE footprint of USC raster (~473-m resolution) ###########################

    # Select Layer by Attribute (Data Management Tools): Select all points OUTSIDE of USC footprint
    arcpy.management.SelectLayerByAttribute(
        in_layer_or_view=layer,
        # "SWITCH_SELECTION" de-selects all points in pre-existing selection + then selects all points in feature layer that were excluded from pre-existing selection (* exact inverse of selection created by 'Select Layer By Location' tool)
        selection_type="SWITCH_SELECTION"
    )

    outside_count = int(arcpy.management.GetCount(layer)[0])

    # Calculate Field (Data Management Tools): Populate "native_raster_resolution_m" field of selected rows (outside USC fp) with '473' (CalculateField writes data to original feature class file for ONLY rows held in feature layer selection)
    arcpy.management.CalculateField(
        in_table=layer,
        field=field_name,
        expression=473,
        expression_type="PYTHON3"
    )

    ########################### reset resolution to Null for points with NO bathy/slope data ###########################

    # Select Layer by Attribute (Data Management Tools): Clear selection so update cursor below sees all rows
    arcpy.management.SelectLayerByAttribute(
        in_layer_or_view=layer,
        selection_type="CLEAR_SELECTION"
    )

    # fix points falling on the tiny gap between GBC and USC rasters that have calculated NO depth/slope value but still recieved a 10/473 label in code above - set resolution back to Null where BOTH bathy and slope are missing (no raster was actually used for that point)
    nulled_count = 0
    with arcpy.da.UpdateCursor(in_points, ["bathymetric_depth_m", "bathymetric_slope_deg", field_name]) as cursor:
        # loop through each row
        for row in cursor:
            # treat both None and NaN as missing for depth and slope
            depth_missing = row[0] is None or (isinstance(row[0], float) and math.isnan(row[0]))
            slope_missing = row[1] is None or (isinstance(row[1], float) and math.isnan(row[1]))
            # if BOTH depth AND slope are missing, blank out the resolution value
            if depth_missing and slope_missing:
                row[2] = None
                nulled_count += 1
                # write blanked value back to row
                cursor.updateRow(row)


    ########################### check data populated in "native_raster_resolution_m" field ###########################

    # count total n records (rows) in feature class table
    total = int(arcpy.management.GetCount(in_points)[0])

    # print report header with input point feature class name + counts of rows with each raster resolution
    print(f"\n------ Report for {os.path.basename(in_points)} ------\n")
    print(f"  Total rows in Attribute Table: {total}")
    print(f"  Rows INSIDE USC footprint (10 m): {inside_count}")
    print(f"  Rows OUTSIDE USC footprint (473 m): {outside_count}")
    # report rows reset to Null because they had no bathy/slope data (no raster used)
    print(f"  Rows reset to Null (no bathy/slope data): {nulled_count}")

    # check for any rows in "native_raster_resolution_m" that were NOT populated (still Null)
    null_count = 0
    null_oids = []
    with arcpy.da.SearchCursor(in_points, ["OID@", field_name]) as cursor:
        for oid, value in cursor:
            if value is None:
                null_count += 1
                null_oids.append(oid)
    if null_count == nulled_count:
        print(f"  '{field_name}' field: ✅ All rows populated (or set to Null if NO calculated bathy OR slope value)")
    else:
        print(f"  '{field_name}' field: ⚠️[WARNING] {null_count} unpopulated (Null) value at OID(s): {null_oids}")

    # report all unique values present in "native_raster_resolution_m" *should ouput [10, 473, None]*
    unique_vals = sorted({row[0] for row in arcpy.da.SearchCursor(in_points, [field_name])}, key=lambda x: (x is None, x))
    print(f"  Unique values in '{field_name}' field: {unique_vals} \n")

    # delete temporary feature layer object
    arcpy.management.Delete(layer)


# Call function for all point feature classes
assign_raster_resolution(hf_points_p26920, usc_fp_p26920_v6642)
assign_raster_resolution(sperm_points_p26920, usc_fp_p26920_v6642)
assign_raster_resolution(odont_points_p26920, usc_fp_p26920_v6642)


------ Report for calc_HF_background_points_p26920 ------

  Total rows in Attribute Table: 4319
  Rows INSIDE USC footprint (10 m): 3665
  Rows OUTSIDE USC footprint (473 m): 654
  Rows reset to Null (no bathy/slope data): 0
  'native_raster_resolution_m' field: ✅ All rows populated (or set to Null if NO calculated bathy OR slope value)
  Unique values in 'native_raster_resolution_m' field: [10, 473] 


------ Report for calc_sperm_points_p26920 ------

  Total rows in Attribute Table: 358
  Rows INSIDE USC footprint (10 m): 242
  Rows OUTSIDE USC footprint (473 m): 116
  Rows reset to Null (no bathy/slope data): 0
  'native_raster_resolution_m' field: ✅ All rows populated (or set to Null if NO calculated bathy OR slope value)
  Unique values in 'native_raster_resolution_m' field: [10, 473] 


------ Report for calc_odontocetes_points_p26920 ------

  Total rows in Attribute Table: 590
  Rows INSIDE USC footprint (10 m): 492
  Rows OUTSIDE USC footprint (473 m): 98
  Rows reset to Nu